In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/MedicalNLP_RAG")

print("ROOT exists:", ROOT.exists())
print("ROOT:", ROOT)

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/MedicalNLP_RAG")

# -----------------------------
# Validation
# -----------------------------
VAL_GROUNDING = (
    ROOT / "results" / "phase4C_grounding_v2" / "tables"
    / "question_grounding_v2.csv"
)

VAL_SCORES = (
    ROOT / "results" / "phase4B_evidence_filtering" / "tables"
    / "phase4B_answer_scores_per_question.csv"
)

# -----------------------------
# Locked internal test
# -----------------------------
TEST_GROUNDING = (
    ROOT / "results" / "phase5_one_time_locked_internal_test"
    / "stageA_label_blind" / "question_grounding_v2.csv"
)

TEST_SCORES = (
    ROOT / "results" / "phase5_one_time_locked_internal_test"
    / "tables" / "locked_test_answer_scores_per_question.csv"
)

paths = {
    "VAL_GROUNDING": VAL_GROUNDING,
    "VAL_SCORES": VAL_SCORES,
    "TEST_GROUNDING": TEST_GROUNDING,
    "TEST_SCORES": TEST_SCORES,
}

for name, path in paths.items():
    print(f"{name}: {path.exists()}  ->  {path}")

In [ ]:
val_grounding = pd.read_csv(VAL_GROUNDING, dtype={"id": str})
val_scores = pd.read_csv(VAL_SCORES, dtype={"id": str})

test_grounding = pd.read_csv(TEST_GROUNDING, dtype={"id": str})
test_scores = pd.read_csv(TEST_SCORES, dtype={"id": str})

print("\nSHAPES")
print("Validation grounding:", val_grounding.shape)
print("Validation scores   :", val_scores.shape)
print("Test grounding      :", test_grounding.shape)
print("Test scores         :", test_scores.shape)

print("\nVALIDATION GROUNDING COLUMNS")
print(val_grounding.columns.tolist())

print("\nVALIDATION SCORE COLUMNS")
print(val_scores.columns.tolist())

print("\nTEST GROUNDING COLUMNS")
print(test_grounding.columns.tolist())

print("\nTEST SCORE COLUMNS")
print(test_scores.columns.tolist())

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# --------------------------------------------------
# 1. Önce koşul ve görev adlarını kontrol et
# --------------------------------------------------
print("Validation grounding conditions:",
      val_grounding["condition"].unique().tolist())

print("Validation score conditions:",
      val_scores["condition"].unique().tolist())

print("Test grounding conditions:",
      test_grounding["condition"].unique().tolist())

print("Test score conditions:",
      test_scores["condition"].unique().tolist())

print("\nValidation types:",
      val_grounding["type"].unique().tolist())

print("Test types:",
      test_grounding["type"].unique().tolist())


# --------------------------------------------------
# 2. Validation: frozen Document Top-5 koşulunu seç
# --------------------------------------------------
vg = val_grounding[
    val_grounding["condition"].astype(str).str.lower().eq("document_top5")
].copy()

vs = val_scores[
    val_scores["condition"].astype(str).str.lower().eq("document_top5")
].copy()

print("\nFiltered validation grounding:", vg.shape)
print("Filtered validation scores   :", vs.shape)

assert len(vg) == 862, f"Expected 862 grounding rows, got {len(vg)}"
assert len(vs) == 862, f"Expected 862 score rows, got {len(vs)}"


# --------------------------------------------------
# 3. Validation dosyasında ortak task_score oluştur
# --------------------------------------------------
def make_task_score(row):
    qtype = str(row["type"]).lower()

    if qtype in ["yesno", "yes/no"]:
        return row["yesno_correct"]

    elif qtype == "factoid":
        return row["factoid_mrr"]

    elif qtype == "list":
        return row["list_f1"]

    elif qtype == "summary":
        return row["rougeL_f"]

    return np.nan


vs["task_score"] = vs.apply(make_task_score, axis=1)


# --------------------------------------------------
# 4. Validation grounding + quality birleştir
# --------------------------------------------------
val_merged = vg.merge(
    vs[["id", "type", "condition", "task_score"]],
    on=["id", "type", "condition"],
    how="inner",
    validate="one_to_one"
)

print("Validation merged:", val_merged.shape)

assert len(val_merged) == 862


# --------------------------------------------------
# 5. Locked test grounding + quality birleştir
# --------------------------------------------------
# Locked test zaten final frozen condition içeriyor.
# condition eşleşmesinde olası isim farklılıklarını önlemek için id + type kullanıyoruz.

test_merged = test_grounding.merge(
    test_scores[["id", "type", "task_score"]],
    on=["id", "type"],
    how="inner",
    validate="one_to_one"
)

print("Locked-test merged:", test_merged.shape)

assert len(test_merged) == 860


# --------------------------------------------------
# 6. Task-specific Spearman correlation
# --------------------------------------------------
TASK_ORDER = ["yesno", "factoid", "list", "summary"]

def task_correlations(df, split_name):
    rows = []

    for task in TASK_ORDER:
        d = df[df["type"].astype(str).str.lower() == task].copy()

        d["mean_top1_entailment"] = pd.to_numeric(
            d["mean_top1_entailment"], errors="coerce"
        )
        d["task_score"] = pd.to_numeric(
            d["task_score"], errors="coerce"
        )

        d = d.dropna(
            subset=["mean_top1_entailment", "task_score"]
        )

        if len(d) < 3:
            rho, p = np.nan, np.nan
        else:
            rho, p = spearmanr(
                d["mean_top1_entailment"],
                d["task_score"]
            )

        rows.append({
            "split": split_name,
            "task": task,
            "n": len(d),
            "grounding_metric": "mean_top1_entailment",
            "task_metric": {
                "yesno": "correctness",
                "factoid": "MRR",
                "list": "F1",
                "summary": "ROUGE-L"
            }[task],
            "spearman_rho": rho,
            "p_value": p
        })

    return pd.DataFrame(rows)


val_corr = task_correlations(val_merged, "Validation")
test_corr = task_correlations(test_merged, "Locked test")

corr_table = pd.concat(
    [val_corr, test_corr],
    ignore_index=True
)

print("\nTASK-SPECIFIC GROUNDING–QUALITY CORRELATIONS")
display(corr_table)

# Kaydet
OUT_DIR = (
    ROOT / "results" /
    "reviewer_revision_grounding_quality"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

out_file = OUT_DIR / "task_specific_grounding_quality_correlations.csv"
corr_table.to_csv(out_file, index=False)

print("\nSaved:", out_file)

In [ ]:
from statsmodels.stats.multitest import multipletests

corr_holm = corr_table.copy()

corr_holm["holm_p"] = np.nan

for split in corr_holm["split"].unique():
    mask = corr_holm["split"] == split

    raw_p = corr_holm.loc[mask, "p_value"].values

    reject, p_adj, _, _ = multipletests(
        raw_p,
        alpha=0.05,
        method="holm"
    )

    corr_holm.loc[mask, "holm_p"] = p_adj
    corr_holm.loc[mask, "holm_significant"] = reject

corr_holm["holm_significant"] = (
    corr_holm["holm_significant"].astype(bool)
)

display(
    corr_holm[
        [
            "split",
            "task",
            "n",
            "task_metric",
            "spearman_rho",
            "p_value",
            "holm_p",
            "holm_significant"
        ]
    ]
)

out_file = (
    OUT_DIR /
    "task_specific_grounding_quality_correlations_holm.csv"
)

corr_holm.to_csv(out_file, index=False)

print("Saved:", out_file)

In [ ]:
from statsmodels.stats.multitest import multipletests
from scipy.stats import spearmanr
import numpy as np
import pandas as pd

GROUNDING_METRICS = [
    "mean_top1_entailment",
    "top1_supported_claim_rate",
    "mean_top3_max_entailment",
    "top3_supported_claim_rate",
]

TASK_METRIC_NAMES = {
    "yesno": "Correctness",
    "factoid": "MRR",
    "list": "F1",
    "summary": "ROUGE-L",
}

TASK_ORDER = ["yesno", "factoid", "list", "summary"]


def grounding_sensitivity_correlations(df, split_name):
    rows = []

    for task in TASK_ORDER:
        d_task = df[
            df["type"].astype(str).str.lower().eq(task)
        ].copy()

        for grounding_metric in GROUNDING_METRICS:

            d = d_task[
                [grounding_metric, "task_score"]
            ].copy()

            d[grounding_metric] = pd.to_numeric(
                d[grounding_metric],
                errors="coerce"
            )

            d["task_score"] = pd.to_numeric(
                d["task_score"],
                errors="coerce"
            )

            d = d.dropna()

            if len(d) < 3:
                rho, p = np.nan, np.nan
            elif d[grounding_metric].nunique() < 2:
                rho, p = np.nan, np.nan
            elif d["task_score"].nunique() < 2:
                rho, p = np.nan, np.nan
            else:
                rho, p = spearmanr(
                    d[grounding_metric],
                    d["task_score"]
                )

            rows.append({
                "split": split_name,
                "task": task,
                "n": len(d),
                "task_metric": TASK_METRIC_NAMES[task],
                "grounding_metric": grounding_metric,
                "spearman_rho": rho,
                "p_value": p,
            })

    return pd.DataFrame(rows)


# -----------------------------------------
# Validation + locked test
# -----------------------------------------

val_sens = grounding_sensitivity_correlations(
    val_merged,
    "Validation"
)

test_sens = grounding_sensitivity_correlations(
    test_merged,
    "Locked test"
)

sens_table = pd.concat(
    [val_sens, test_sens],
    ignore_index=True
)


# -----------------------------------------
# Conservative Holm correction:
# 16 sensitivity tests within each split
# -----------------------------------------

sens_table["holm_p_within_split"] = np.nan
sens_table["holm_significant_within_split"] = False

for split in sens_table["split"].unique():

    mask = (
        (sens_table["split"] == split)
        & sens_table["p_value"].notna()
    )

    pvals = sens_table.loc[
        mask, "p_value"
    ].to_numpy()

    reject, p_adj, _, _ = multipletests(
        pvals,
        alpha=0.05,
        method="holm"
    )

    sens_table.loc[
        mask, "holm_p_within_split"
    ] = p_adj

    sens_table.loc[
        mask, "holm_significant_within_split"
    ] = reject


# -----------------------------------------
# Sonuç
# -----------------------------------------

display(
    sens_table[
        [
            "split",
            "task",
            "task_metric",
            "grounding_metric",
            "n",
            "spearman_rho",
            "p_value",
            "holm_p_within_split",
            "holm_significant_within_split",
        ]
    ]
)


# -----------------------------------------
# Kaydet
# -----------------------------------------

out_file = (
    OUT_DIR /
    "grounding_quality_sensitivity_correlations.csv"
)

sens_table.to_csv(
    out_file,
    index=False
)

print("\nSaved:", out_file)

In [ ]:
from pathlib import Path
import json

ROOT = Path("/content/drive/MyDrive/MedicalNLP_RAG")

# Phase4B veya evidence/filtering adını taşıyan notebookları bul
candidates = []

for p in ROOT.rglob("*.ipynb"):
    name = p.name.lower()
    if (
        "phase4b" in name
        or ("evidence" in name and "filter" in name)
        or ("prun" in name)
    ):
        candidates.append(p)

print("Candidate notebooks:")
for i, p in enumerate(candidates, 1):
    print(i, p)

In [ ]:
search_terms = [
    "sentence_filtered",
    "sentence-filtered",
    "sentence filter",
    "sentences",
    "top_k_sent",
    "sentence_score",
]

for nb_path in candidates:
    print("\n" + "=" * 100)
    print("NOTEBOOK:", nb_path)
    print("=" * 100)

    with open(nb_path, "r", encoding="utf-8") as f:
        nb = json.load(f)

    found = False

    for idx, cell in enumerate(nb.get("cells", [])):
        src = "".join(cell.get("source", []))

        if any(term.lower() in src.lower() for term in search_terms):
            found = True
            print(f"\n--- CELL {idx} ---")
            print(src[:8000])

    if not found:
        print("No matching cells found.")

In [ ]:
from pathlib import Path
import shutil
import json
import hashlib
import pandas as pd

ROOT = Path("/content/drive/MyDrive/MedicalNLP_RAG")
REPO = ROOT / "repository_package"

# Temiz başlangıç
if REPO.exists():
    shutil.rmtree(REPO)

for sub in [
    "splits",
    "prompts",
    "analysis",
    "outputs/validation",
    "outputs/locked_test",
    "reviewer_revision",
    "manifests",
    "data",
]:
    (REPO / sub).mkdir(parents=True, exist_ok=True)

print("Repository folder created:")
print(REPO)

In [ ]:
# --------------------------------------------
# 1. Mevcut notebookları bul
# --------------------------------------------

notebook_patterns = {
    "phase4b_evidence_filtering": "*Phase4B*Evidence*Filtering*.ipynb",
    "phase4c_grounding_faithfulness": "*Phase4C*Grounding*Faithfulness*.ipynb",
    "phase4d_confidence_calibration": "*Phase4D*Confidence*Calibration*.ipynb",
    "phase4e_selective_answering": "*Phase4E*Selective*Answering*.ipynb",
    "phase5_locked_internal_test": "*Phase5*Locked*Internal*Test*.ipynb",
    "phase6_final_analysis": "*Phase6*Final*Analysis*.ipynb",
    "revision_grounding_quality_analysis": "revision1.ipynb",
}

found_notebooks = {}

for label, pattern in notebook_patterns.items():
    matches = list(ROOT.rglob(pattern))
    found_notebooks[label] = matches

print("NOTEBOOKS")
for label, matches in found_notebooks.items():
    print(f"\n{label}")
    for p in matches:
        print("  ", p)

In [ ]:
# --------------------------------------------
# 2. Soru bazlı mevcut çıktı dosyalarını ara
# --------------------------------------------

search_names = [
    "question_grounding_v2.csv",
    "phase4B_answer_scores_per_question.csv",
    "locked_test_answer_scores_per_question.csv",
    "task_specific_grounding_quality_correlations.csv",
    "task_specific_grounding_quality_correlations_holm.csv",
    "grounding_quality_sensitivity_correlations.csv",
]

print("\nOUTPUT FILES")

found_outputs = {}

for fname in search_names:
    matches = list(ROOT.rglob(fname))
    found_outputs[fname] = matches

    print(f"\n{fname}")
    for p in matches:
        print("  ", p)

In [ ]:
# --------------------------------------------
# 3. Split, manifest ve prompt dosyalarını ara
# --------------------------------------------

keywords = [
    "split",
    "manifest",
    "checksum",
    "prompt",
    "freeze",
]

candidate_aux = []

for p in ROOT.rglob("*"):
    if p.is_file():
        name = p.name.lower()

        if any(k in name for k in keywords):
            candidate_aux.append(p)

print("\nAUXILIARY FILES")

for p in candidate_aux[:200]:
    print(p)

print("\nTotal auxiliary candidates:", len(candidate_aux))

In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/MedicalNLP_RAG")

keywords = [
    "confidence",
    "oof",
    "prediction",
    "selective",
    "policy",
    "mask",
    "per_question",
    "question_level",
]

matches = []

for p in ROOT.rglob("*"):
    if not p.is_file():
        continue

    name = p.name.lower()

    if p.suffix.lower() in {".csv", ".json", ".jsonl", ".parquet"}:
        if any(k in name for k in keywords):
            matches.append(p)

for p in sorted(matches):
    print(p)

print("\nTotal:", len(matches))

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/MedicalNLP_RAG")

AUDIT_FILES = [
    ROOT / "results/phase4A_answer_generation/tables/validation_answer_scores_per_question.csv",
    ROOT / "results/phase4B_evidence_filtering/tables/phase4B_answer_scores_per_question.csv",
    ROOT / "results/phase4C_grounding_v2/tables/question_grounding_v2.csv",
    ROOT / "results/phase4D_v2_task_specific_confidence/tables/task_specific_oof_predictions.csv",

    ROOT / "results/phase5_one_time_locked_internal_test/stageA_label_blind/question_grounding_v2.csv",
    ROOT / "results/phase5_one_time_locked_internal_test/stageA_label_blind/frozen_policy_selection_masks.csv",
    ROOT / "results/phase5_one_time_locked_internal_test/stageA_label_blind/locked_test_confidence_features.csv",
    ROOT / "results/phase5_one_time_locked_internal_test/stageA_label_blind/locked_test_frozen_confidence_predictions.csv",

    ROOT / "results/phase5_one_time_locked_internal_test/tables/locked_test_answer_scores_per_question.csv",
    ROOT / "results/phase5_one_time_locked_internal_test/tables/locked_test_generation_predictions.csv",
    ROOT / "results/phase5_one_time_locked_internal_test/tables/locked_test_retrieval_per_question.csv",
]

for path in AUDIT_FILES:
    print("\n" + "=" * 100)
    print(path)
    print("=" * 100)

    if not path.exists():
        print("MISSING")
        continue

    df = pd.read_csv(path, nrows=3)

    print("Columns:")
    print(df.columns.tolist())

    suspicious = [
        c for c in df.columns
        if any(
            k in c.lower()
            for k in [
                "question",
                "body",
                "gold",
                "ideal_answer",
                "exact_answer",
                "abstract",
                "title",
                "evidence_text",
                "raw"
            ]
        )
    ]

    print("Potentially sensitive / redistributable-content columns:")
    print(suspicious)

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/MedicalNLP_RAG")

patterns = [
    "*validation_862q_3conditions*.csv",
    "*document_top5*.csv",
    "*generation*.csv",
    "*predictions*.csv",
]

found = set()

for pattern in patterns:
    for p in ROOT.rglob(pattern):
        if p.is_file() and "phase4" in str(p).lower():
            found.add(p)

for p in sorted(found):
    print(p)